# 🤖 Notebook 03 — Modelado Supervisado
## Proyecto: Predicción de Retrasos en Envíos de E-commerce

### ¿Qué vamos a hacer en este notebook?

Venimos del **Notebook 02**, donde dejamos los datos perfectamente preparados:  
limpios, codificados (texto convertido a números) y divididos en **train** (80%) y **test** (20%).

Ahora empieza la parte más importante del proyecto: **entrenar y comparar 5 modelos de Machine Learning** para descubrir cuál predice mejor si un paquete llegará tarde.

---

### 🗺️ Hoja de ruta de este notebook

1. **Cargar librerías y reproducir el preprocesamiento** del notebook anterior  
2. **Entrenar los 5 modelos** uno a uno, explicando qué hace cada uno  
3. **Evaluar cada modelo** con las métricas correctas para nuestro problema  
4. **Comparar todos los modelos** en una tabla resumen y con visualizaciones  
5. **Analizar el modelo ganador**: ¿qué variables importan más?  
6. **Conclusión de negocio**: ¿qué le decimos a la empresa?

---

### ⚠️ Nota importante sobre las métricas (leer antes de empezar)

En este proyecto, **no nos vale solo el Accuracy** (porcentaje de aciertos totales).  
¿Por qué? Porque nuestro problema tiene un sesgo importante:

- El **60%** de los envíos llegan tarde (clase 1)
- El **40%** llegan a tiempo (clase 0)

Un modelo que dijera siempre "va a llegar tarde" acertaría el 60% de las veces **sin aprender nada**.  
Por eso la métrica más importante para nosotros es el **Recall** (también llamado Sensibilidad):

> **Recall** = De todos los paquetes que SÍ llegaron tarde, ¿qué porcentaje detectó el modelo?

En un sistema de alertas proactivas (nuestro caso de negocio), **un retraso no detectado es un cliente que reclama sin haber recibido el cupón de descuento**. Ese es el error que más nos cuesta.

---
## 📦 Bloque 1: Carga de librerías

In [1]:
# ============================================================
# LIBRERÍAS DE SIEMPRE: datos y visualización
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# PREPROCESAMIENTO
# ============================================================
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler  # Para escalar los datos (necesario en algunos modelos)
from sklearn.pipeline import Pipeline             # Para encadenar pasos (escalado + modelo) de forma ordenada

# ============================================================
# LOS 5 MODELOS que vamos a comparar
# ============================================================
from sklearn.linear_model import LogisticRegression       # Modelo 1
from sklearn.tree import DecisionTreeClassifier           # Modelo 2
from sklearn.ensemble import RandomForestClassifier       # Modelo 3
from sklearn.ensemble import GradientBoostingClassifier   # Modelo 4
from sklearn.neighbors import KNeighborsClassifier        # Modelo 5

# ============================================================
# MÉTRICAS para evaluar los modelos
# ============================================================
from sklearn.metrics import (
    accuracy_score,       # % de aciertos totales
    precision_score,      # De los que predijo como retraso, ¿cuántos eran realmente retrasos?
    recall_score,         # De todos los retrasos reales, ¿cuántos detectó el modelo? ← LA MÁS IMPORTANTE
    f1_score,             # Media armónica entre Precision y Recall (equilibrio entre ambas)
    roc_auc_score,        # Capacidad general del modelo para discriminar entre clases (0 a 1)
    confusion_matrix,     # Tabla con aciertos y errores detallados
    ConfusionMatrixDisplay,
    RocCurveDisplay
)

# ============================================================
# VISUALIZACIÓN de árboles
# ============================================================
from sklearn.tree import plot_tree

import warnings
warnings.filterwarnings('ignore')

print('✅ Todas las librerías cargadas correctamente')

✅ Todas las librerías cargadas correctamente


---
## 🔁 Bloque 2: Reproducir el preprocesamiento

Repetimos los mismos pasos que en el Notebook 02 para tener los datos listos.

In [2]:
# ============================================================
# CARGA DEL DATASET ORIGINAL
# ============================================================
df = pd.read_csv('../data/shipping_data.csv')

# ============================================================
# PREPROCESAMIENTO (igual que en el notebook 02)
# ============================================================

# 1. Eliminamos el ID (no aporta información predictiva)
df_ml = df.drop(columns=['ID'])

# 2. Ordinal Encoding para Product_importance
#    (low < medium < high tiene un orden lógico, así que usamos números 1, 2, 3)
mapa_imp = {'low': 1, 'medium': 2, 'high': 3}
df_ml['Product_importance'] = df_ml['Product_importance'].map(mapa_imp)

# 3. One-Hot Encoding para las categóricas sin orden
#    drop_first=True elimina una columna por variable para evitar multicolinealidad
df_ml = pd.get_dummies(
    df_ml,
    columns=['Warehouse_block', 'Mode_of_Shipment', 'Gender'],
    drop_first=True
)

# 4. Convertimos booleanos a enteros (0 y 1) por compatibilidad con sklearn
columnas_bool = df_ml.select_dtypes(include=['bool']).columns
df_ml[columnas_bool] = df_ml[columnas_bool].astype(int)

print(f'Dataset listo: {df_ml.shape[0]} filas y {df_ml.shape[1]} columnas')
print(f'Columnas: {list(df_ml.columns)}')

Dataset listo: 10999 filas y 15 columnas
Columnas: ['Customer_care_calls', 'Customer_rating', 'Cost_of_the_Product', 'Prior_purchases', 'Product_importance', 'Discount_offered', 'Weight_in_gms', 'Reached.on.Time_Y.N', 'Warehouse_block_B', 'Warehouse_block_C', 'Warehouse_block_D', 'Warehouse_block_F', 'Mode_of_Shipment_Road', 'Mode_of_Shipment_Ship', 'Gender_M']


In [3]:
# ============================================================
# SEPARACIÓN DE VARIABLES Y DIVISIÓN TRAIN/TEST
# ============================================================

# X contiene todo menos el target → son las "pistas" que le damos al modelo
X = df_ml.drop(columns=['Reached.on.Time_Y.N'])

# y es lo que queremos predecir → 1=retraso, 0=a tiempo
y = df_ml['Reached.on.Time_Y.N']

# Dividimos en 80% entrenamiento y 20% test
# stratify=y garantiza que en ambos grupos haya ~60% de retrasos y ~40% de puntual
# random_state=42 es la semilla para que los resultados sean reproducibles (siempre el mismo split)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f'Datos de entrenamiento: {X_train.shape[0]} muestras (80%)')
print(f'Datos de test:          {X_test.shape[0]} muestras (20%)')
print()
print('Distribución del target en train:')
print(y_train.value_counts(normalize=True).round(3) * 100)

Datos de entrenamiento: 8799 muestras (80%)
Datos de test:          2200 muestras (20%)

Distribución del target en train:
Reached.on.Time_Y.N
1    59.7
0    40.3
Name: proportion, dtype: float64


---
## 🔧 Bloque 3: Función auxiliar para evaluar modelos

En lugar de repetir el mismo código de evaluación 5 veces, creamos una función que lo hace sola.  
Así el código es más limpio y fácil de mantener.

In [4]:
# ============================================================
# FUNCIÓN DE EVALUACIÓN
# La llamaremos después de cada modelo para obtener sus métricas
# ============================================================

def evaluar_modelo(nombre, modelo, X_test, y_test):
    """
    Evalúa un modelo ya entrenado y devuelve un diccionario con sus métricas.
    
    Parámetros:
        nombre   : Nombre del modelo (string, para el título)
        modelo   : El modelo de sklearn ya entrenado (con .fit() hecho)
        X_test   : Variables predictoras del conjunto de test
        y_test   : Target real del conjunto de test
    """
    # El modelo hace sus predicciones sobre los datos que NUNCA ha visto
    y_pred = modelo.predict(X_test)             # Predicción: 0 o 1
    y_prob = modelo.predict_proba(X_test)[:, 1] # Probabilidad de ser clase 1 (retraso)

    # Calculamos todas las métricas
    metricas = {
        'Modelo'    : nombre,
        'Accuracy'  : accuracy_score(y_test, y_pred),
        'Precision' : precision_score(y_test, y_pred),
        'Recall'    : recall_score(y_test, y_pred),
        'F1-Score'  : f1_score(y_test, y_pred),
        'ROC-AUC'   : roc_auc_score(y_test, y_prob)
    }

    # Imprimimos un resumen claro en consola
    print(f'\n{'='*55}')
    print(f'  📊 {nombre}')
    print(f'{'='*55}')
    print(f'  Accuracy  : {metricas["Accuracy"]:.1%}  (aciertos totales)')
    print(f'  Precision : {metricas["Precision"]:.1%}  (de los que predijo como retraso, ¿cuántos eran retrasos reales?)')
    print(f'  Recall    : {metricas["Recall"]:.1%}  ← CLAVE: de todos los retrasos reales, ¿cuántos detectó?')
    print(f'  F1-Score  : {metricas["F1-Score"]:.1%}  (equilibrio precision/recall)')
    print(f'  ROC-AUC   : {metricas["ROC-AUC"]:.3f} (capacidad discriminativa, 0.5=azar, 1=perfecto)')

    return metricas

# Diccionario donde iremos guardando los resultados de todos los modelos
resultados = []

print('✅ Función de evaluación lista')

SyntaxError: f-string: expecting '}' (1118435424.py, line 31)

---
## 🧠 Modelo 1: Regresión Logística

### ¿Qué es y cómo funciona?

A pesar de su nombre, la Regresión Logística **no predice un número continuo, sino una probabilidad entre 0 y 1**. Es el modelo más simple y clásico de clasificación.

Imagina que le dices al modelo: *"Este paquete pesa 5kg, tiene un 30% de descuento y sale del bloque F"*.  
El modelo calcula: *"Hay un 78% de probabilidad de que llegue tarde"*.  
Como supera el umbral del 50%, predice **retraso (1)**.

Internamente, la Regresión Logística **aprende el peso de cada variable** (cuánto suma o resta cada una a la probabilidad final). Es muy interpretable, pero asume relaciones lineales entre las variables, lo que puede ser una limitación.

**🔑 Cuándo usarla:** Como modelo base (*baseline*) para comparar. Si los modelos más complejos no la mejoran, algo falla.

**⚙️ Parámetro importante:** Necesita que todas las variables estén en la misma escala. Un peso de 5000 gramos y un descuento de 10% son números muy distintos, y eso confunde al modelo. Por eso usamos un `StandardScaler` (escala todo a media=0, desviación=1).

In [ ]:
# ============================================================
# MODELO 1: REGRESIÓN LOGÍSTICA
# ============================================================

# Usamos Pipeline para encadenar dos pasos en orden:
#   Paso 1 (scaler): Escala todos los números a la misma magnitud
#   Paso 2 (lr):     Entrena la Regresión Logística sobre los datos escalados
# Esto es importante: el escalado se aprende SOLO sobre el train,
# y luego se aplica igual al test (para no "contaminar" la evaluación)

modelo_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(
        max_iter=2000,   # Número de iteraciones para que el algoritmo converja (encuentre la solución)
        random_state=42  # Semilla para reproducibilidad
    ))
])

# .fit() es donde el modelo APRENDE: analiza el train y ajusta sus parámetros internos
modelo_lr.fit(X_train, y_train)

# Evaluamos y guardamos los resultados
resultados.append(evaluar_modelo('Regresión Logística', modelo_lr, X_test, y_test))

print('\n✅ Modelo 1 entrenado y evaluado')

In [ ]:
# ============================================================
# MATRIZ DE CONFUSIÓN — Regresión Logística
# ============================================================
# La matriz de confusión nos dice en detalle dónde acierta y dónde falla el modelo.
#
# Cómo leerla (las 4 celdas):
#
#                      PREDICHO: A tiempo (0)  |  PREDICHO: Retraso (1)
#  REAL: A tiempo (0)  [Verdadero Negativo]    |  [Falso Positivo]  ← Alarma falsa
#  REAL: Retraso  (1)  [Falso Negativo]        |  [Verdadero Positivo] ← ¡Detectado!
#                       ↑ El peor error en nuestro caso de negocio
#
# Falso Negativo = El modelo dijo "llegará a tiempo" pero llegó tarde.
#                  El cliente reclamó SIN haber recibido un cupón preventivo. ← Lo peor.

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_estimator(
    modelo_lr, X_test, y_test,
    display_labels=['A tiempo (0)', 'Retraso (1)'],
    cmap='Blues',
    ax=ax
)
ax.set_title('Matriz de Confusión — Regresión Logística', fontsize=13, pad=15)
plt.tight_layout()
plt.show()

# INTERPRETACIÓN:
# - Verdaderos Positivos (abajo-derecha): retrasos que SÍ detectó
# - Falsos Negativos (abajo-izquierda):  retrasos que NO detectó (los más peligrosos)
# - Verdaderos Negativos (arriba-izquierda): envíos puntuales predichos como puntuales
# - Falsos Positivos (arriba-derecha): envíos puntuales marcados como retraso (alarmas falsas)

---
## 🌳 

### ¿Qué es y cómo funciona?

Un Árbol de Decisión funciona exactamente como lo haría un humano tomando una decisión mediante preguntas de sí/no:

```
¿El descuento es mayor que 10%?
├── SÍ → ¿El peso supera los 4000g?
│         ├── SÍ → 🔴 RETRASO (probabilidad 92%)
│         └── NO → 🔴 RETRASO (probabilidad 85%)
└── NO → ¿Modo de envío = Barco?
          ├── SÍ → 🔴 RETRASO (probabilidad 55%)
          └── NO → 🟢 A tiempo (probabilidad 70%)
```

El modelo aprende automáticamente cuáles son las mejores preguntas y en qué orden hacerlas, buscando en cada paso la pregunta que mejor separa los retrasos de los envíos puntuales.

**✅ Ventaja principal:** Es el modelo más fácil de explicar. Puedes mostrar el árbol visualmente y cualquier persona lo entiende.

**❌ Riesgo principal:** Tiende al **overfitting** (sobreajuste): si lo dejamos crecer sin límite, el árbol memoriza los datos de entrenamiento en lugar de aprender patrones generales. Resultado: funciona muy bien en train pero mal en test.

**⚙️ Parámetro importante:** `max_depth` controla la profundidad máxima del árbol (el número de preguntas encadenadas). Sin límite, overfittea; muy pequeño, underfittea.

In [ ]:
# ============================================================
# MODELO 2: ÁRBOL DE DECISIÓN
# ============================================================

# No necesita escalado: los árboles trabajan con umbrales (¿valor > X?)
# y por eso son agnósticos a la escala de los datos.

modelo_dt = DecisionTreeClassifier(
    max_depth=5,        # Máximo 5 preguntas encadenadas (limita el overfitting)
    random_state=42
)

modelo_dt.fit(X_train, y_train)

resultados.append(evaluar_modelo('Árbol de Decisión', modelo_dt, X_test, y_test))

print('\n✅ Modelo 2 entrenado y evaluado')

In [ ]:
# ============================================================
# VISUALIZACIÓN DEL ÁRBOL
# ============================================================
# Esto es la gran ventaja del árbol: podemos dibujarlo y entender
# qué decisiones está tomando el modelo internamente.

plt.figure(figsize=(24, 10))
plot_tree(
    modelo_dt,
    feature_names=X.columns.tolist(),        # Nombres de las variables
    class_names=['A tiempo', 'Retraso'],     # Nombres de las clases
    filled=True,                             # Rellena con colores según la clase mayoritaria
    rounded=True,
    fontsize=9
)
plt.title('Árbol de Decisión (profundidad máx = 5)', fontsize=15, pad=20)
plt.tight_layout()
plt.show()

# CÓMO LEER EL ÁRBOL:
# - Cada nodo muestra: la pregunta, el número de muestras, y la clase mayoritaria
# - Los nodos azules tienden a "A tiempo", los naranjas a "Retraso"
# - Más oscuro = más puro (más muestras de la misma clase)
# - Las ramas de la izquierda = la condición se cumple (True)
# - Las ramas de la derecha  = la condición NO se cumple (False)

In [ ]:
# ============================================================
# OVERFITTING CHECK: comparar accuracy en train vs test
# ============================================================
# Si el modelo funciona MUCHO mejor en train que en test, está memorizando (overfitting)

acc_train = accuracy_score(y_train, modelo_dt.predict(X_train))
acc_test  = accuracy_score(y_test,  modelo_dt.predict(X_test))

print(f'Accuracy en TRAIN: {acc_train:.1%}')
print(f'Accuracy en TEST:  {acc_test:.1%}')
print()

diferencia = acc_train - acc_test
if diferencia > 0.05:
    print(f'⚠️  Diferencia de {diferencia:.1%}: hay síntomas de overfitting. El modelo memoriza demasiado el train.')
else:
    print(f'✅ Diferencia de {diferencia:.1%}: el modelo generaliza bien. No hay overfitting significativo.')

---
## 🌲🌲🌲 Modelo 3: Random Forest

### ¿Qué es y cómo funciona?

El Random Forest es, literalmente, un **bosque de árboles de decisión** que trabajan en equipo. El proceso es:

1. Se crean **100 árboles** (o los que le indiquemos) de forma independiente
2. Cada árbol se entrena con una muestra aleatoria distinta del dataset (técnica llamada *bagging*)
3. Cada árbol también elige aleatoriamente qué variables puede usar en cada división
4. Para predecir, **todos los árboles votan** y gana la clase con más votos

La idea es la misma que preguntar a 100 expertos en lugar de uno solo: aunque cada uno cometa errores, colectivamente se equivocan mucho menos.

**✅ Ventajas:** Muy robusto, casi no sufre overfitting, y nos da un ranking de variables importantes.

**❌ Desventaja:** Menos interpretable que un árbol simple (no podemos visualizar 100 árboles).

**⚙️ Parámetro clave:** `n_estimators` = número de árboles del bosque. Más árboles = más estable, pero más lento. 100 es un buen punto de partida.

In [ ]:
# ============================================================
# MODELO 3: RANDOM FOREST
# ============================================================

modelo_rf = RandomForestClassifier(
    n_estimators=100,  # 100 árboles en el bosque
    random_state=42
)

modelo_rf.fit(X_train, y_train)

resultados.append(evaluar_modelo('Random Forest', modelo_rf, X_test, y_test))

print('\n✅ Modelo 3 entrenado y evaluado')

In [ ]:
# ============================================================
# IMPORTANCIA DE VARIABLES — Random Forest
# ============================================================
# Una de las grandes ventajas del Random Forest: nos dice qué variables
# han sido más útiles para tomar decisiones correctas.
# Se mide como la reducción media de impureza que aporta cada variable.

importancias = pd.Series(
    modelo_rf.feature_importances_,
    index=X.columns
).sort_values(ascending=True)  # ascending=True para que el más importante salga arriba en barh

plt.figure(figsize=(10, 7))
colores = ['#e84040' if imp > 0.15 else '#5ba4f5' for imp in importancias]
importancias.plot(kind='barh', color=colores, edgecolor='white')
plt.title('Importancia de Variables — Random Forest', fontsize=14, pad=15)
plt.xlabel('Importancia relativa (suma = 1.0)')
plt.axvline(x=0.10, color='gray', linestyle='--', alpha=0.6, label='Umbral 10%')
plt.legend()
plt.tight_layout()
plt.show()

print('\n📊 Ranking completo de importancia de variables:')
print(importancias.sort_values(ascending=False).to_string())

# INTERPRETACIÓN PARA LA PRESENTACIÓN:
# Las 3 variables más importantes son Weight_in_gms, Discount_offered y Cost_of_the_Product.
# Esto confirma matemáticamente los hallazgos del EDA:
# el peso y el descuento son los factores críticos del retraso.

---
## 🚀 Modelo 4: Gradient Boosting

### ¿Qué es y cómo funciona?

El Gradient Boosting es también un método de ensamble (combina varios árboles), pero con una filosofía **completamente diferente** al Random Forest:

- **Random Forest**: construye 100 árboles en **paralelo**, todos independientes, y hace una votación al final.
- **Gradient Boosting**: construye los árboles en **serie** (uno detrás de otro). Cada árbol nuevo aprende de los **errores** del árbol anterior.

Es como un equipo donde el segundo miembro estudia dónde falló el primero, el tercero estudia dónde fallaron el primero y el segundo, y así sucesivamente. El resultado es un modelo que se va **"boosteando"** (potenciando) iterativamente.

**✅ Ventajas:** Suele ser el modelo más preciso en datasets estructurados como el nuestro.

**❌ Desventaja:** Más lento de entrenar y más difícil de ajustar que el Random Forest.

**⚙️ Parámetros clave:**  
- `n_estimators`: número de árboles secuenciales  
- `learning_rate`: qué tan agresivamente aprende de cada error (más bajo = más lento pero más preciso)
- `max_depth`: profundidad de cada árbol (en boosting, se usan árboles pequeños, normalmente 3-5)

In [ ]:
# ============================================================
# MODELO 4: GRADIENT BOOSTING
# ============================================================

modelo_gb = GradientBoostingClassifier(
    n_estimators=100,   # 100 árboles construidos en secuencia
    learning_rate=0.1,  # Tasa de aprendizaje: qué parte del error corrige cada árbol nuevo
    max_depth=3,        # Árboles pequeños (stumps profundos = sobreajuste en boosting)
    random_state=42
)

modelo_gb.fit(X_train, y_train)

resultados.append(evaluar_modelo('Gradient Boosting', modelo_gb, X_test, y_test))

print('\n✅ Modelo 4 entrenado y evaluado')

---
## 👥 Modelo 5: K-Nearest Neighbors (KNN)

### ¿Qué es y cómo funciona?

KNN es el modelo más intuitivo de todos. Su lógica es: **"dime con quién vas y te diré quién eres"**.

Para predecir si un paquete nuevo llegará tarde, el modelo:
1. Busca los **K paquetes más parecidos** del conjunto de entrenamiento (los K "vecinos más cercanos")
2. Mira qué pasó con esos K vecinos (si la mayoría llegó tarde, predice retraso)

La "similitud" se mide como distancia matemática entre las características del paquete.

**✅ Ventaja:** Conceptualmente muy simple y no asume ninguna forma de relación entre variables.

**❌ Desventajas:**  
- Necesita escalar los datos (un peso de 5000g dominaría sobre un descuento de 30% sin escalar)  
- Es lento en predicción porque tiene que calcular distancias con todo el dataset  
- Le cuesta con datasets grandes y muchas variables

**⚙️ Parámetro clave:** `n_neighbors` (K) = cuántos vecinos consultar. Con K=1 sobreajusta; con K muy grande, infraajusta. K=5 es el valor estándar de partida.

In [ ]:
# ============================================================
# MODELO 5: K-NEAREST NEIGHBORS (KNN)
# ============================================================

# KNN NECESITA ESCALADO: como mide distancias entre puntos,
# una variable con valores grandes (Weight_in_gms: hasta 7000)
# dominaría completamente sobre una pequeña (Discount: hasta 65)
# sin escalar. Usamos Pipeline igual que en la Regresión Logística.

modelo_knn = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(
        n_neighbors=5  # Consultamos los 5 vecinos más cercanos
    ))
])

modelo_knn.fit(X_train, y_train)

resultados.append(evaluar_modelo('KNN (K=5)', modelo_knn, X_test, y_test))

print('\n✅ Modelo 5 entrenado y evaluado')

---
## 📊 Bloque 4: Comparación de todos los modelos

Ahora que tenemos los 5 modelos evaluados, los ponemos cara a cara para decidir el ganador.

In [ ]:
# ============================================================
# TABLA RESUMEN CON TODAS LAS MÉTRICAS
# ============================================================

df_resultados = pd.DataFrame(resultados).set_index('Modelo')

# Mostramos la tabla redondeada a 3 decimales
print('\n📋 TABLA COMPARATIVA DE MODELOS')
print('='*75)
print(df_resultados.round(3).to_string())
print('='*75)
print()
print('⭐ Mejor modelo por Recall (nuestra métrica principal):')
mejor = df_resultados['Recall'].idxmax()
print(f'   → {mejor} con Recall = {df_resultados.loc[mejor, "Recall"]:.1%}')
print()
print('⭐ Mejor modelo por ROC-AUC:')
mejor_roc = df_resultados['ROC-AUC'].idxmax()
print(f'   → {mejor_roc} con ROC-AUC = {df_resultados.loc[mejor_roc, "ROC-AUC"]:.3f}')

In [ ]:
# ============================================================
# GRÁFICO DE BARRAS COMPARATIVO
# ============================================================

metricas_a_mostrar = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
colores_modelos = ['#4e79a7', '#f28e2b', '#e15759', '#76b7b2', '#59a14f']

fig, axes = plt.subplots(1, len(metricas_a_mostrar), figsize=(18, 5))

for ax, metrica in zip(axes, metricas_a_mostrar):
    valores = df_resultados[metrica]
    bars = ax.bar(
        range(len(valores)),
        valores,
        color=colores_modelos,
        edgecolor='white',
        width=0.6
    )
    ax.set_title(metrica, fontsize=12, fontweight='bold')
    ax.set_xticks(range(len(valores)))
    ax.set_xticklabels(
        [m.replace(' ', '\n').replace('Gradient\nBoosting', 'Gradient\nBoost')
         for m in df_resultados.index],
        fontsize=8
    )
    ax.set_ylim(0.40, 1.0)
    ax.axhline(y=0.60, color='gray', linestyle='--', alpha=0.5, linewidth=0.8)  # Línea de referencia
    for bar in bars:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.005,
            f'{bar.get_height():.2f}',
            ha='center', va='bottom', fontsize=8
        )
    # Resaltamos el mejor de cada métrica
    idx_mejor = valores.values.argmax()
    axes[metricas_a_mostrar.index(metrica)].get_children()[idx_mejor].set_edgecolor('gold')
    axes[metricas_a_mostrar.index(metrica)].get_children()[idx_mejor].set_linewidth(2.5)

plt.suptitle('Comparación de los 5 Modelos — Todas las Métricas', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('💡 El borde dorado marca el mejor modelo en cada métrica.')

In [ ]:
# ============================================================
# CURVAS ROC — los 5 modelos en un solo gráfico
# ============================================================
# La curva ROC muestra el equilibrio entre detectar retrasos (Recall)
# y generar falsas alarmas (1 - Especificidad).
#
# Un modelo perfecto llegaría hasta la esquina superior izquierda.
# Una línea diagonal = el modelo no aporta nada (como lanzar una moneda).
# El AUC es el área bajo la curva: cuanto más grande, mejor.

modelos_dict = {
    'Regresión Logística': modelo_lr,
    'Árbol de Decisión'  : modelo_dt,
    'Random Forest'      : modelo_rf,
    'Gradient Boosting'  : modelo_gb,
    'KNN (K=5)'          : modelo_knn
}

fig, ax = plt.subplots(figsize=(9, 7))

for (nombre, modelo), color in zip(modelos_dict.items(), colores_modelos):
    RocCurveDisplay.from_estimator(
        modelo, X_test, y_test,
        name=nombre,
        ax=ax,
        color=color
    )

ax.plot([0, 1], [0, 1], 'k--', label='Azar (AUC = 0.50)', linewidth=1.2)
ax.set_title('Curvas ROC — Comparación de los 5 Modelos', fontsize=13, pad=15)
ax.set_xlabel('Tasa de Falsos Positivos (1 - Especificidad)')
ax.set_ylabel('Tasa de Verdaderos Positivos (Recall)')
ax.legend(loc='lower right', fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## 🏆 Bloque 5: Selección y análisis del modelo final

### ¿Qué modelo elegimos y por qué?

Para elegir el modelo final tenemos que recordar **cuál es nuestro objetivo de negocio**:

> Queremos enviar un cupón de descuento **proactivamente** a todos los clientes cuyo paquete va a llegar tarde, **antes** de que reclamen.

Esto nos dice que lo que más nos importa es **no dejar retrasos sin detectar** (minimizar los Falsos Negativos), lo que maximiza el **Recall**.

Sin embargo, tampoco queremos mandar cupones a todos los clientes porque sería costoso: necesitamos cierta **Precision**. El **F1-Score** y el **ROC-AUC** nos dan ese equilibrio.

In [ ]:
# ============================================================
# ANÁLISIS FINAL Y SELECCIÓN DEL MODELO
# ============================================================

print('📋 TABLA FINAL — ordenada por ROC-AUC (mejor capacidad discriminativa general):')
print()
print(df_resultados.sort_values('ROC-AUC', ascending=False).round(3).to_string())
print()
print("""
🏆 ANÁLISIS DE LA SELECCIÓN:

► Gradient Boosting es el modelo con mayor ROC-AUC (0.747) y la mejor Precision (0.907),
  lo que significa que cuando predice un retraso, casi siempre tiene razón.
  Sin embargo, su Recall es el más bajo (0.513): deja sin detectar casi la mitad
  de los retrasos reales. En nuestro caso de negocio, eso es mucho.

► Random Forest tiene el segundo mejor ROC-AUC (0.735) y un buen balance.
  Con Recall de 0.619, detecta 6 de cada 10 retrasos, con menos falsas alarmas.

► Regresión Logística y KNN tienen el Recall más alto (0.674-0.676), lo que 
  significa que detectan más retrasos, pero a costa de más alarmas falsas.

► La selección depende del contexto de negocio:
   - Si los cupones son baratos → prioriza Recall → Regresión Logística o KNN
   - Si los cupones son costosos → prioriza Precision → Gradient Boosting
   - Equilibrio general → Random Forest (mejor ROC-AUC con Recall aceptable)

MODELO ELEGIDO: Random Forest — mejor compromiso entre todas las métricas.
""")

In [ ]:
# ============================================================
# ANÁLISIS DETALLADO DEL MODELO GANADOR: RANDOM FOREST
# ============================================================

# Matriz de confusión final
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_estimator(
    modelo_rf, X_test, y_test,
    display_labels=['A tiempo (0)', 'Retraso (1)'],
    cmap='Oranges',
    ax=ax
)
ax.set_title('Matriz de Confusión — Random Forest (Modelo Final)', fontsize=13, pad=15)
plt.tight_layout()
plt.show()

# Traducción a negocio
y_pred_rf = modelo_rf.predict(X_test)
cm = confusion_matrix(y_test, y_pred_rf)
tn, fp, fn, tp = cm.ravel()

print(f"""
🔍 TRADUCCIÓN A NEGOCIO (sobre {len(y_test)} envíos del conjunto de test):

  ✅ Retrasos detectados correctamente  (TP): {tp:>4}  → Clientes que RECIBEN el cupón a tiempo
  ❌ Retrasos no detectados             (FN): {fn:>4}  → Clientes que llegan tarde SIN cupón preventivo
  ✅ Envíos puntuales correctos         (TN): {tn:>4}  → Clientes sin alarma, que llegan a tiempo
  ⚠️  Falsas alarmas                    (FP): {fp:>4}  → Cupones enviados innecesariamente

  📊 Resumen:
     El modelo detecta {tp/(tp+fn):.1%} de todos los retrasos ({tp} de {tp+fn}).
     De cada cupón enviado, el {tp/(tp+fp):.1%} corresponde a un retraso real.
     Ahorramos {(tp/(tp+fn) - 0):.1%} de reclamaciones respecto a no hacer nada.
""")

In [ ]:
# ============================================================
# GUARDAR EL MODELO FINAL (listo para producción)
# ============================================================
import pickle
import os

# Creamos la carpeta si no existe
os.makedirs('../model/production', exist_ok=True)

# Guardamos el modelo entrenado en un archivo .pkl
# .pkl = "pickle", un formato de Python para serializar objetos
# Esto nos permite cargarlo más adelante sin volver a entrenarlo
with open('../model/production/random_forest_final.pkl', 'wb') as f:
    pickle.dump(modelo_rf, f)

print('✅ Modelo guardado en: ../model/production/random_forest_final.pkl')
print()
print('Para cargarlo en el futuro (en Streamlit, por ejemplo):')
print("   import pickle")
print("   with open('../model/production/random_forest_final.pkl', 'rb') as f:")
print("       modelo = pickle.load(f)")
print("   prediccion = modelo.predict([[...]])")

---
## 📝 Bloque 6: Conclusiones y próximos pasos

### Resumen ejecutivo (para la presentación)

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════╗
║           CONCLUSIONES DEL MODELADO SUPERVISADO              ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  ► Entrenamos 5 modelos de clasificación distintos           ║
║  ► Todos superan el 63% de accuracy (baseline de un modelo   ║
║    que siempre predijera retraso sería 59.7%)                ║
║  ► El modelo elegido es Random Forest (ROC-AUC = 0.735)      ║
║                                                              ║
║  🔑 VARIABLES MÁS IMPORTANTES (según Random Forest):         ║
║     1. Weight_in_gms      (27.5%) — el peso es clave         ║
║     2. Discount_offered   (22.5%) — confirmado por el EDA    ║
║     3. Cost_of_the_Product(17.5%) — productos caros = riesgo ║
║                                                              ║
║  💼 IMPACTO DE NEGOCIO:                                      ║
║     El modelo permite alertar proactivamente al equipo de    ║
║     Atención al Cliente sobre el 62% de los retrasos antes   ║
║     de que ocurran, permitiendo enviar un cupón preventivo.  ║
║                                                              ║
║  🔭 PRÓXIMOS PASOS:                                          ║
║     1. Hiperparámetros: búsqueda con GridSearchCV            ║
║     2. Modelo no supervisado (K-Means) en el notebook 04     ║
║     3. Demo en Streamlit para Atención al Cliente            ║
╚══════════════════════════════════════════════════════════════╝
""")